# AIkenGPT 2.5B — 重み読込と生成テスト

このノートブックは、AIkenGPT 2.5Bの`.pt`チェックポイントから
`model_state_dict`だけを読み込み、テキスト生成を試すためのものです。

- 学習処理なし
- optimizer読込なし
- LoRA / PEFTなし
- GPT-2 tokenizer（`vocab_size=50257`）
- KV cache付き生成

最初にColabのランタイムをGPUへ変更し、設定セルの
`CHECKPOINT_PATH`だけ確認してください。


In [ ]:
# 必要なライブラリ
%pip install -q tiktoken


In [ ]:
# Google Driveと実行環境
from google.colab import drive
drive.mount("/content/drive")

import gc
import random
from dataclasses import dataclass
from pathlib import Path

import tiktoken
import torch
import torch.nn as nn
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPUが有効ではありません。"
        "Colabの「ランタイムのタイプを変更」からGPUを選択してください。"
    )

GPU_NAME = torch.cuda.get_device_name(0)
COMPUTE_CAPABILITY = torch.cuda.get_device_capability(0)
INFERENCE_DTYPE = (
    torch.bfloat16
    if COMPUTE_CAPABILITY[0] >= 8
    else torch.float16
)

print("PyTorch:", torch.__version__)
print("GPU:", GPU_NAME)
print("Compute capability:", COMPUTE_CAPABILITY)
print("Inference dtype:", INFERENCE_DTYPE)


Mounted at /content/drive
PyTorch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-80GB
Compute capability: (8, 0)
Inference dtype: torch.bfloat16


## 1. 設定

`CHECKPOINT_PATH`を、テストしたい`.pt`ファイルへ変更します。

`PROMPT_STYLE`は次の3種類です。

- `"qa"`: G検定CPTモデル向け
- `"sft"`: Instruction Tuningモデル向け
- `"plain"`: 事前学習モデル向け


In [ ]:
# ===== 変更する場所 =====
CHECKPOINT_PATH = (
      "/content/drive/MyDrive/checkpoint_escape/checkpoint_054000.pt"#####変更
)

PROMPT_STYLE = "plain"  # "qa", "sft", "plain"
SYSTEM_PROMPT = (
    "あなたはAIと機械学習について正確に説明するアシスタントです。"
)

TEST_QUESTIONS = [
    "検証データにデータリーケージがあった場合、どのような影響がありますか。",
    "機械学習における過学習と、その代表的な対策を説明してください。",
    "TransformerにおけるAttention機構を説明してください。",
]

MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.7
TOP_K = 50
TOP_P = 0.9
RANDOM_SEED = 1337

# ===== モデル構造（学習時と同一）=====
@dataclass
class ModelConfig:
    embedding_dim: int = 2560
    hidden_dim: int = 10240
    num_attention_heads: int = 20
    layer_count: int = 30
    rope_theta: float = 1_000_000.0
    vocab_size: int = 50257
    max_sequence_length: int = 2048


config = ModelConfig()

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

checkpoint_path = Path(CHECKPOINT_PATH)
if not checkpoint_path.is_file():
    raise FileNotFoundError(
        f"チェックポイントが見つかりません: {checkpoint_path}"
    )

print("Checkpoint:", checkpoint_path)
print(
    "File size:",
    f"{checkpoint_path.stat().st_size / 1024**3:.2f} GB",
)
print("Prompt style:", PROMPT_STYLE)


Checkpoint: /content/drive/MyDrive/checkpoint_escape/checkpoint_054000.pt
File size: 29.25 GB
Prompt style: plain


## 2. AIkenGPT 2.5Bモデル定義


In [ ]:
class TokenEmbedding(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding_table = nn.Embedding(
            config.vocab_size,
            config.embedding_dim,
        )

    def forward(self, input_indices):
        return self.token_embedding_table(input_indices)


class RotaryEmbedding(nn.Module):
    def __init__(
        self,
        dim,
        max_seq_len=2048,
        rope_theta=1_000_000.0,
    ):
        super().__init__()

        inv_freq = 1.0 / (
            rope_theta
            ** (
                torch.arange(
                    0,
                    dim,
                    2,
                    dtype=torch.float32,
                )
                / dim
            )
        )
        positions = torch.arange(
            max_seq_len,
            dtype=torch.float32,
        )
        freqs = torch.einsum(
            "i,j->ij",
            positions,
            inv_freq,
        )
        freqs_complex = torch.polar(
            torch.ones_like(freqs),
            freqs,
        )
        self.register_buffer(
            "freqs_complex",
            freqs_complex.view(
                1,
                1,
                max_seq_len,
                -1,
            ),
            persistent=False,
        )

    def apply_rotary_emb(
        self,
        x,
        position_offset=0,
    ):
        sequence_length = x.size(2)
        end_position = position_offset + sequence_length

        if end_position > self.freqs_complex.size(2):
            raise RuntimeError(
                "RoPEの最大系列長を超えました。"
            )

        x_complex = torch.view_as_complex(
            x.float().reshape(
                *x.shape[:-1],
                -1,
                2,
            )
        )
        freqs = self.freqs_complex[
            :,
            :,
            position_offset:end_position,
            :,
        ]
        x_rotated = x_complex * freqs

        return torch.view_as_real(
            x_rotated
        ).reshape_as(x).to(x.dtype)


class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.num_heads = config.num_attention_heads
        self.embed_dim = config.embedding_dim
        self.head_dim = (
            self.embed_dim // self.num_heads
        )

        self.query_fc = nn.Linear(
            self.embed_dim,
            self.embed_dim,
            bias=False,
        )
        self.key_fc = nn.Linear(
            self.embed_dim,
            self.embed_dim,
            bias=False,
        )
        self.value_fc = nn.Linear(
            self.embed_dim,
            self.embed_dim,
            bias=False,
        )
        self.output_projection = nn.Linear(
            self.embed_dim,
            self.embed_dim,
        )

        self.rotary_emb = RotaryEmbedding(
            dim=self.head_dim,
            max_seq_len=config.max_sequence_length,
            rope_theta=config.rope_theta,
        )

        self.register_buffer(
            "cache_k",
            None,
            persistent=False,
        )
        self.register_buffer(
            "cache_v",
            None,
            persistent=False,
        )
        self.current_pos = 0

    def _project(self, x, position_offset):
        batch_size, token_length, channels = x.shape

        query = self.query_fc(x)
        key = self.key_fc(x)
        value = self.value_fc(x)

        query = query.view(
            batch_size,
            token_length,
            self.num_heads,
            self.head_dim,
        ).transpose(1, 2)
        key = key.view(
            batch_size,
            token_length,
            self.num_heads,
            self.head_dim,
        ).transpose(1, 2)
        value = value.view(
            batch_size,
            token_length,
            self.num_heads,
            self.head_dim,
        ).transpose(1, 2)

        query = self.rotary_emb.apply_rotary_emb(
            query,
            position_offset=position_offset,
        )
        key = self.rotary_emb.apply_rotary_emb(
            key,
            position_offset=position_offset,
        )

        return query, key, value, channels

    def forward(self, x, use_cache=False):
        if not use_cache:
            query, key, value, channels = self._project(
                x,
                position_offset=0,
            )
            output = F.scaled_dot_product_attention(
                query,
                key,
                value,
                attn_mask=None,
                is_causal=True,
            )
        elif self.cache_k is None:
            query, key, value, channels = self._project(
                x,
                position_offset=0,
            )
            token_length = x.size(1)

            if token_length > self.config.max_sequence_length:
                raise RuntimeError(
                    "入力がmax_sequence_lengthを超えています。"
                )

            self.cache_k = torch.zeros(
                x.size(0),
                self.num_heads,
                self.config.max_sequence_length,
                self.head_dim,
                device=x.device,
                dtype=key.dtype,
            )
            self.cache_v = torch.zeros(
                x.size(0),
                self.num_heads,
                self.config.max_sequence_length,
                self.head_dim,
                device=x.device,
                dtype=value.dtype,
            )
            self.cache_k[:, :, :token_length, :] = key
            self.cache_v[:, :, :token_length, :] = value
            self.current_pos = token_length

            output = F.scaled_dot_product_attention(
                query,
                key,
                value,
                attn_mask=None,
                is_causal=True,
            )
        else:
            if x.size(1) != 1:
                raise RuntimeError(
                    "KV cache使用中の追加入力は1 tokenである必要があります。"
                )

            if (
                self.current_pos
                >= self.config.max_sequence_length
            ):
                raise RuntimeError(
                    "KV cacheの最大系列長を超えました。"
                )

            query, key, value, channels = self._project(
                x,
                position_offset=self.current_pos,
            )
            self.cache_k[
                :,
                :,
                self.current_pos:self.current_pos + 1,
                :,
            ] = key
            self.cache_v[
                :,
                :,
                self.current_pos:self.current_pos + 1,
                :,
            ] = value
            self.current_pos += 1

            output = F.scaled_dot_product_attention(
                query,
                self.cache_k[:, :, :self.current_pos, :],
                self.cache_v[:, :, :self.current_pos, :],
                attn_mask=None,
                is_causal=False,
            )

        output = output.transpose(
            1,
            2,
        ).contiguous().view(
            x.size(0),
            x.size(1),
            channels,
        )
        return self.output_projection(output)

    def reset_cache(self):
        self.cache_k = None
        self.cache_v = None
        self.current_pos = 0


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(
                config.embedding_dim,
                config.hidden_dim,
                bias=False,
            ),
            nn.ReLU(),
            nn.Linear(
                config.hidden_dim,
                config.embedding_dim,
                bias=False,
            ),
        )

    def forward(self, input_tensor):
        return self.net(input_tensor)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.layer_norm1 = nn.LayerNorm(
            config.embedding_dim
        )
        self.layer_norm2 = nn.LayerNorm(
            config.embedding_dim
        )
        self.multihead_attention = MultiHeadAttention(
            config=config
        )
        self.feed_forward = FeedForward(
            config=config
        )

    def forward(self, input_tensor, use_cache=False):
        attention_output = self.multihead_attention(
            self.layer_norm1(input_tensor),
            use_cache=use_cache,
        )
        residual_attention = (
            input_tensor + attention_output
        )
        feedforward_output = self.feed_forward(
            self.layer_norm2(residual_attention)
        )
        return residual_attention + feedforward_output


class VocabularyLogits(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.output_norm = nn.LayerNorm(
            config.embedding_dim
        )
        self.vocab_projection = nn.Linear(
            config.embedding_dim,
            config.vocab_size,
            bias=False,
        )

    def forward(self, transformer_block_output):
        return self.vocab_projection(
            self.output_norm(
                transformer_block_output
            )
        )


class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding_layer = TokenEmbedding(
            config=config
        )
        self.blocks = nn.ModuleList(
            [
                TransformerBlock(config=config)
                for _ in range(config.layer_count)
            ]
        )
        self.vocab_projection = VocabularyLogits(
            config=config
        )
        self.criterion = nn.CrossEntropyLoss()

    def forward(
        self,
        input_indices,
        target_indices=None,
        use_cache=False,
    ):
        x = self.token_embedding_layer(input_indices)

        for block in self.blocks:
            x = block(
                x,
                use_cache=use_cache,
            )

        logits = self.vocab_projection(x)

        if target_indices is None:
            return logits, None

        loss = self.criterion(
            logits.reshape(
                -1,
                logits.size(-1),
            ),
            target_indices.reshape(-1),
        )
        return logits, loss

    def reset_cache(self):
        for block in self.blocks:
            block.multihead_attention.reset_cache()

    @torch.inference_mode()
    def generate(
        self,
        input_indices,
        max_new_tokens,
        temperature=0.7,
        top_k=50,
        top_p=0.9,
        stop_ids=None,
    ):
        self.eval()
        self.reset_cache()
        stop_ids = set(stop_ids or [])
        next_input = input_indices
        generated_ids = []

        for _ in range(max_new_tokens):
            logits, _ = self.forward(
                next_input,
                target_indices=None,
                use_cache=True,
            )
            last_logits = logits[:, -1, :]

            if temperature <= 0:
                next_token = torch.argmax(
                    last_logits,
                    dim=-1,
                    keepdim=True,
                )
            else:
                last_logits = (
                    last_logits / temperature
                )

                if top_k is not None:
                    effective_top_k = min(
                        top_k,
                        last_logits.size(-1),
                    )
                    threshold = torch.topk(
                        last_logits,
                        effective_top_k,
                    ).values[:, -1, None]
                    last_logits = torch.where(
                        last_logits < threshold,
                        torch.full_like(
                            last_logits,
                            float("-inf"),
                        ),
                        last_logits,
                    )

                if top_p is not None:
                    sorted_logits, sorted_indices = (
                        torch.sort(
                            last_logits,
                            descending=True,
                        )
                    )
                    sorted_probs = F.softmax(
                        sorted_logits,
                        dim=-1,
                    )
                    cumulative_probs = torch.cumsum(
                        sorted_probs,
                        dim=-1,
                    )
                    remove_mask = (
                        cumulative_probs > top_p
                    )
                    remove_mask[..., 1:] = (
                        remove_mask[..., :-1].clone()
                    )
                    remove_mask[..., 0] = False
                    sorted_logits = (
                        sorted_logits.masked_fill(
                            remove_mask,
                            float("-inf"),
                        )
                    )
                    last_logits = torch.full_like(
                        last_logits,
                        float("-inf"),
                    ).scatter(
                        -1,
                        sorted_indices,
                        sorted_logits,
                    )

                probabilities = F.softmax(
                    last_logits,
                    dim=-1,
                )
                next_token = torch.multinomial(
                    probabilities,
                    num_samples=1,
                )

            token_id = int(next_token.item())
            if token_id in stop_ids:
                break

            generated_ids.append(token_id)
            next_input = next_token

        self.reset_cache()
        return generated_ids


# このmodelを次のセルでもそのまま使用し、巨大モデルを二重に構築しない。
model = GPT(config)
parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)
print(f"Model definition ready: {parameter_count / 1e9:.3f}B parameters")


Model definition ready: 2.617B parameters


## 3. `.pt`から重みだけ読み込む

`optimizer_state_dict`などはGPUへ読み込みません。
`mmap=True`と`assign=True`を使い、CPUメモリ上の余分なコピーを抑えます。


In [ ]:
device = torch.device("cuda")
torch.set_float32_matmul_precision("high")

print("チェックポイントを読み込んでいます...")
try:
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        mmap=True,
        weights_only=False,
    )
except TypeError:
    # 古いPyTorch向け
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
    )

if (
    isinstance(checkpoint, dict)
    and "model_state_dict" in checkpoint
):
    state_dict = checkpoint["model_state_dict"]
elif (
    isinstance(checkpoint, dict)
    and checkpoint
    and all(
        torch.is_tensor(value)
        for value in checkpoint.values()
    )
):
    state_dict = checkpoint
else:
    raise KeyError(
        "model_state_dictが見つかりません。"
        "このノートブックはAIkenGPTの.ptチェックポイント専用です。"
    )

clean_state_dict = {
    key.removeprefix("_orig_mod."): value
    for key, value in state_dict.items()
}

try:
    load_result = model.load_state_dict(
        clean_state_dict,
        strict=True,
        assign=True,
    )
except TypeError:
    load_result = model.load_state_dict(
        clean_state_dict,
        strict=True,
    )

if load_result.missing_keys:
    raise RuntimeError(
        f"Missing keys: {load_result.missing_keys}"
    )
if load_result.unexpected_keys:
    raise RuntimeError(
        f"Unexpected keys: {load_result.unexpected_keys}"
    )

checkpoint_metadata = {
    key: checkpoint.get(key)
    for key in (
        "current_step",
        "completed_cpt_steps",
        "completed_sft_steps",
    )
    if isinstance(checkpoint, dict)
    and key in checkpoint
}

print("推論用dtypeへ変換し、GPUへ移動しています...")

# model.to(dtype=...)はcomplex64のRoPEバッファまで変換してしまう。
# そのため、浮動小数点の学習パラメータだけを変換する。
for parameter in model.parameters():
    if parameter.is_floating_point():
        parameter.data = parameter.data.to(
            dtype=INFERENCE_DTYPE
        )

# RoPEのcomplex64バッファはdtypeを保ったままGPUへ移動する。
model = model.to(device=device)
model.eval()

del checkpoint
del state_dict
del clean_state_dict
gc.collect()
torch.cuda.empty_cache()

allocated_gb = (
    torch.cuda.memory_allocated() / 1024**3
)

print("Model weights loaded successfully.")
print("Metadata:", checkpoint_metadata or "none")
print(
    "GPU memory allocated:",
    f"{allocated_gb:.2f} GB",
)


チェックポイントを読み込んでいます...
推論用dtypeへ変換し、GPUへ移動しています...
Model weights loaded successfully.
Metadata: {'current_step': 54000}
GPU memory allocated: 4.91 GB


## 4. 生成関数


In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
EOS_TOKEN_ID = 50256


def build_prompt(question, style=PROMPT_STYLE):
    if style == "qa":
        return f"質問: {question}\n解説:"

    if style == "sft":
        system_part = (
            f"### システム:\n{SYSTEM_PROMPT}\n\n"
            if SYSTEM_PROMPT
            else ""
        )
        return (
            f"{system_part}"
            f"### ユーザー:\n{question}\n\n"
            "### アシスタント:\n"
        )

    if style == "plain":
        return question

    raise ValueError(
        'PROMPT_STYLEは"qa", "sft", "plain"のいずれかです。'
    )


def generate_text(
    question,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
):
    prompt = build_prompt(question)
    prompt_ids = tokenizer.encode(
        prompt,
        disallowed_special=(),
    )

    available_tokens = (
        config.max_sequence_length
        - len(prompt_ids)
    )
    actual_max_new_tokens = min(
        max_new_tokens,
        available_tokens,
    )

    if actual_max_new_tokens <= 0:
        raise ValueError(
            "プロンプトが最大系列長を超えています。"
        )

    input_indices = torch.tensor(
        [prompt_ids],
        dtype=torch.long,
        device=device,
    )

    generated_ids = model.generate(
        input_indices=input_indices,
        max_new_tokens=actual_max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        stop_ids={EOS_TOKEN_ID},
    )

    generated_text = tokenizer.decode(
        generated_ids,
        errors="replace",
    )

    return {
        "prompt": prompt,
        "generated_text": generated_text,
        "prompt_tokens": len(prompt_ids),
        "generated_tokens": len(generated_ids),
    }


print("Generation function ready.")


Generation function ready.


## 5. 生成テスト


In [ ]:
PROMPT_STYLE = "plain"  # "qa", "sft", "plain"
SYSTEM_PROMPT = (
    "あなたはAIと機械学習について正確に説明するアシスタントです。"
)

TEST_QUESTIONS = [
    "検証データにデータリーケージがあった場合、どのような影響がありますか。",
    "機械学習における過学習と、その代表的な対策を説明してください。",
    "TransformerにおけるAttention機構を説明してください。",
]

MAX_NEW_TOKENS = 2048
TEMPERATURE = 8.0
TOP_K = 50
TOP_P = 0.9
RANDOM_SEED = 1337

In [ ]:
for index, question in enumerate(
    TEST_QUESTIONS,
    start=1,
):
    result = generate_text(question)

    print("\n" + "=" * 90)
    print(f"TEST {index}")
    print("-" * 90)
    print(result["prompt"], end="")
    print(result["generated_text"])
    print("-" * 90)
    print(
        f'prompt tokens: {result["prompt_tokens"]} | '
        f'generated tokens: {result["generated_tokens"]}'
    )



TEST 1
------------------------------------------------------------------------------------------
検証データにデータリーケージがあった場合、どのような影響がありますか。
影響はありません。
データリーケージとは、データが外部に流出することです。
そのため、データが外部に流出することで、どのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどのような影響がありますか。
データが外部に流出することで、システムにどの

## 6. 任意の質問を1問だけ生成

`question`を書き換えて、このセルだけ再実行できます。


In [ ]:
question = "勾配消失問題について説明してください。"

result = generate_text(question)

print(result["prompt"], end="")
print(result["generated_text"])
print(
    f'\nGenerated tokens: {result["generated_tokens"]}'
)


勾配消失問題について説明してください。
その他の問題は、ほぼ毎日のように問題になります。
これらは私が読んだ本の中では最もよく出てくる問題です。
これらの問題は、数学の知識がある人でも、数学が苦手な人でも、これらの問題について理解していることを前提にしています。
また、これらの問題を理解していることを前提にしていることから、これらの問題について解説している記事もよく読まれています。
そのため、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
そのため、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
そのため、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
そのため、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
このようなことから、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
そのため、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
このようなことから、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
そのため、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
このようなことから、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
このようなことから、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
このようなことから、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
このようなことから、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
そのため、これらの問題についての解説を読んでいる人は、これらの問題について理解していることが前提になっています。
このようなことから、これらの問題についての解説を読んでいる人は、これらの